# Underwriting Orchestrator: Composing Phases 2–7

The orchestrator is the parent LangGraph. Each completed phase is a compiled subgraph added as one node. The parent owns conditional routing, checkpoint persistence, missing-document interrupts, and exception-review interrupts.

```text
START → document_layer ── incomplete → request_missing_documents ─┐
                     └── complete → borrower_analysis             │
                                      ↓                           │
                               property_analysis                  │
                                      ↓                           │
                               property_research                  │
                                      ↓                           │
                           calculations_and_policy                │
                                      ↓                           │
                                reconciliation                    │
                               ↙               ↘                  │
                      exception_review       underwriting_summary │
                               ↓               ↓                  │
                      underwriting_summary → END                  │
                                      ↑___________________________┘
```

The final package is decision support. A qualified human underwriter retains the final lending decision.

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


## 1. Compile the checkpointed parent graph

Subgraphs do not need their own checkpointers. LangGraph propagates the parent's checkpoint context into nested graphs. Every invocation and resume must use the same `thread_id`.

In [ ]:
from underwriting_agent.orchestrator import build_underwriting_orchestrator

orchestrator = build_underwriting_orchestrator(GUIDELINES)
print(orchestrator.get_graph().draw_mermaid())


In [ ]:
from underwriting_agent.intake_packages import resolve_document_paths

def initial_input(loan_id, exclude=None):
    paths = resolve_document_paths(PDF_ROOT, loan_id)
    if exclude:
        paths = [path for path in paths if path.name != exclude]
    return {"loan_id": loan_id, "document_paths": [str(path) for path in paths], "workflow_status": "INTAKE"}

def thread(thread_id):
    return {"configurable": {"thread_id": thread_id}}


## 2. Complete package: automatic analysis and review routing

The Moreno package contains every required document. It runs through all specialist subgraphs automatically, then pauses because its calculated LTV exceeds the demonstration guideline. Complete documents do not imply an approval.

In [ ]:
complete = orchestrator.invoke(initial_input("UW-26-0417-A"), config=thread("notebook-complete"))
print(complete["__interrupt__"][0].value["type"])
print([item["code"] for item in complete["__interrupt__"][0].value["exceptions"]])


## 3. Missing-document pause and resume

We omit the appraisal from input. The graph checkpoints all progress and surfaces an interrupt. Supplying the missing PDF with `Command(resume=...)` restarts the interrupted node, loops back through document intake, and continues the same thread.

In [ ]:
from langgraph.types import Command

missing_config = thread("notebook-missing-appraisal")
paused = orchestrator.invoke(initial_input("UW-26-0417-A", exclude="appraisal_1847_Larkspur.pdf"), config=missing_config)
paused["__interrupt__"][0].value


In [ ]:
resumed = orchestrator.invoke(
    Command(resume={"document_paths": [str(next(path for path in resolve_document_paths(PDF_ROOT, "UW-26-0417-A") if path.name == "appraisal_1847_Larkspur.pdf"))]}),
    config=missing_config,
)
print(resumed["requirements"].complete)
print(resumed["workflow_status"])


## 4. Exception-review pause and resume

The showcase file progresses through every specialist, then pauses with canonical exceptions and proposed conditions. The reviewer may acknowledge them or request changes. This action is recorded in the final package; it is not a lending decision.

In [ ]:
review_config = thread("notebook-exception-review")
paused = orchestrator.invoke(initial_input("WHL-77-2206"), config=review_config)
review_payload = paused["__interrupt__"][0].value
print(review_payload["type"])
print([item["code"] for item in review_payload["exceptions"]])
print(review_payload["conditions"])


In [ ]:
reviewed = orchestrator.invoke(
    Command(resume={"action": "acknowledge", "reviewer": "Notebook Reviewer", "notes": "Reviewed synthetic evidence."}),
    config=review_config,
)
package = reviewed["review_package"]
print(package.review_disposition)
print(package.human_review)
for event in package.observability_log:
    print(event.actor, event.phase, event.action)
print(package.disclaimer)


## 5. Inspect persisted thread state

The latest checkpoint is suitable for a UI status page. A production deployment would replace `InMemorySaver` with a persistent database-backed checkpointer.

In [ ]:
snapshot = orchestrator.get_state(review_config)
print(snapshot.values["loan_id"])
print(snapshot.values["workflow_status"])
print(snapshot.next)


## UI integration boundary

The Streamlit app starts a run with displayed PDF paths and a unique workflow `thread_id`. If `__interrupt__` is present, it renders the upload or review request and sends `Command(resume=...)` under the same thread. The final recommendation includes the human response and the AI-versus-human `observability_log`.

## Optional production services: OpenAI and Pinecone

The parent graph accepts four injected service seams:

- `document_interpreter`: optional OpenAI structured extraction in Phase 2;
- `guideline_store`: optional Pinecone + OpenAI embeddings in Phase 5;
- `review_narrator`: optional OpenAI structured narrative in Phase 7;
- `property_research_service`: optional address-only You.com research in Phase 4B.

If environment flags are off, every seam falls back to deterministic local behavior. SQLite is intentionally not added; the notebook continues to use `InMemorySaver`.

In [ ]:
# Build the parent graph from environment-selected services.
from underwriting_agent.integrations import build_integrations_from_env

integrations = build_integrations_from_env(GUIDELINES, dotenv_path=PROJECT_ROOT / ".env")
service_orchestrator = build_underwriting_orchestrator(
    GUIDELINES,
    document_interpreter=integrations.document_interpreter,
    guideline_store=integrations.guideline_store,
    review_narrator=integrations.review_narrator,
    property_research_service=integrations.property_research_service,
)
print("Document model:", type(integrations.document_interpreter).__name__ if integrations.document_interpreter else "deterministic")
print("Guideline store:", type(integrations.guideline_store).__name__ if integrations.guideline_store else "local")
print("Review narrator:", type(integrations.review_narrator).__name__ if integrations.review_narrator else "deterministic")
print("Property research:", type(integrations.property_research_service).__name__ if integrations.property_research_service else "disabled")
